# Ejercicio 1: Rectificación métrica de superficies

**Asignatura:** Visión Artificial · Máster en IA · Bloque I (Transformaciones planas y homografías)

**Objetivo.** A partir de una fotografía en perspectiva de un suelo con baldosas, calcular la homografía $H$ que genera una **vista cenital** (ortogonal) y comprobar empíricamente que las juntas que *fugaban* en la imagen original son ahora **paralelas**.

**Restricción del enunciado:** todo el núcleo va en **NumPy**, sin OpenCV. OpenCV se usa **solo en el apartado 6** para comparar.

| Apartado | Contenido |
|---|---|
| 0 | Imports y configuración |
| 1 | Carga de la imagen |
| 2 | Puntos origen (4 esquinas del rectángulo real) |
| 3 | Puntos destino (escala proporcional a las medidas reales) |
| 4 | Homografía $H$: DLT + SVD + normalización de Hartley |
| 5 | Warp propio (mapeo inverso + interpolación bilineal) y verificación del paralelismo |
| 6 | Comparación con OpenCV |

## 0. Imports y configuración

> **Trasfondo (Clase 1, "Leyes de la perspectiva" y "Modelo pinhole").** La cámara pinhole proyecta $x = fX/Z,\; y = fY/Z$. Esa transformación *preserva las rectas*, pero **no preserva ángulos ni paralelismo**: las rectas paralelas del mundo convergen en **puntos de fuga**. Por eso las juntas de un suelo se ven convergentes en la foto, y por eso hace falta una transformación proyectiva para deshacerlo.

Cuando el objeto está en un **plano** (el suelo), la relación entre ese plano y la imagen es exactamente una **homografía** (matriz $3\times3$). Esa es la clave del ejercicio.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.set_printoptions(suppress=True, precision=4)
plt.rcParams["figure.dpi"] = 100

# ---------------- CONFIGURACIÓN (edita esto con tus datos) ----------------
IMG_PATH   = "foto_suelo.jpg"   # tu foto. Si no existe, se genera una imagen sintética de demostración
MAX_WIDTH  = 1000               # se reduce la foto a este ancho (el warp en NumPy es costoso con fotos de 4000 px)

# Dimensiones REALES (en cm) del rectángulo cuyas 4 esquinas vas a marcar en la foto
REAL_W_CM  = 240.0              # ancho  (lado TL -> TR)
REAL_H_CM  = 180.0              # alto   (lado TL -> BL)

PX_PER_CM  = 4.0                # escala de la vista cenital: píxeles de salida por cm reales
MARGIN_CM  = 30.0               # margen alrededor del rectángulo en la imagen de salida
# --------------------------------------------------------------------------

## 1. Carga de la imagen

Si no encuentra `foto_suelo.jpg`, el notebook crea una **imagen sintética** (suelo de baldosas de 30 cm visto en perspectiva con una homografía conocida). Sirve para validar el código: como conocemos la $H$ verdadera, podemos comprobar que la recuperamos.

In [ ]:
# ---------- utilidades homogéneas (Clase 1: coordenadas homogéneas) ----------
def homog(p):
    # (N,2) -> (N,3) añadiendo w=1
    p = np.asarray(p, dtype=np.float64)
    return np.hstack([p, np.ones((len(p), 1))])

def inhomog(p):
    # (N,3) -> (N,2) dividiendo por la última coordenada w
    p = np.asarray(p, dtype=np.float64)
    return p[:, :2] / p[:, 2:3]

def apply_H(H, pts):
    # Aplica la homografía H a puntos (N,2) y devuelve (N,2)
    return inhomog((H @ homog(pts).T).T)

In [ ]:
import os

def warp_bilinear(img, H_inv, out_w, out_h):
    # Warp por MAPEO INVERSO con interpolación bilineal, vectorizado.
    # Para cada píxel (x,y) de la imagen de SALIDA calcula de dónde sale en la imagen ORIGEN: p_src = H^-1 · p_dst.
    # Los píxeles de salida que caen fuera de la imagen origen quedan en negro.
    img = np.asarray(img)
    gray = img.ndim == 2
    if gray:
        img = img[..., None]
    h, w, c = img.shape

    xs, ys = np.meshgrid(np.arange(out_w), np.arange(out_h))
    P = np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])      # 3 x N, coordenadas homogéneas
    Q = H_inv @ P
    with np.errstate(divide="ignore", invalid="ignore"):
        u = Q[0] / Q[2]                                           # x en la imagen origen
        v = Q[1] / Q[2]                                           # y en la imagen origen
    valid = np.isfinite(u) & np.isfinite(v) & (u >= 0) & (u <= w - 1) & (v >= 0) & (v <= h - 1)
    u = np.where(valid, u, 0.0)
    v = np.where(valid, v, 0.0)

    x0 = np.clip(np.floor(u).astype(int), 0, w - 2)
    y0 = np.clip(np.floor(v).astype(int), 0, h - 2)
    dx = (u - x0)[:, None]
    dy = (v - y0)[:, None]
    I = img.astype(np.float64)
    out = ((1 - dx) * (1 - dy) * I[y0, x0] + dx * (1 - dy) * I[y0, x0 + 1]
           + (1 - dx) * dy * I[y0 + 1, x0] + dx * dy * I[y0 + 1, x0 + 1])
    out[~valid] = 0
    out = np.clip(np.rint(out), 0, 255).astype(np.uint8).reshape(out_h, out_w, c)
    return out[..., 0] if gray else out

# ---------- datos de demostración (solo se usan si no existe tu foto) ----------
DEMO = not os.path.exists(IMG_PATH)
if DEMO:
    print("No se encuentra", IMG_PATH, "-> usando imagen SINTÉTICA de demostración")
    TILE = 30.0
    W_img, H_img = 1000, 700
    # Homografía verdadera plano(cm) -> imagen(px) definida por 4 correspondencias
    world = np.array([[0, 0], [240, 0], [240, 180], [0, 180]], float)
    cam   = np.array([[300, 140], [720, 170], [950, 610], [60, 590]], float)

    def dlt_simple(src, dst):
        A = []
        for (x, y), (u, v) in zip(src, dst):
            A.append([x, y, 1, 0, 0, 0, -u * x, -u * y, -u])
            A.append([0, 0, 0, x, y, 1, -v * x, -v * y, -v])
        _, _, Vt = np.linalg.svd(np.array(A))
        H = Vt[-1].reshape(3, 3)
        return H / H[2, 2]

    H_true = dlt_simple(world, cam)                       # mundo -> imagen
    # Imagen sintética por mapeo inverso del tablero
    xs, ys = np.meshgrid(np.arange(W_img), np.arange(H_img))
    Q = np.linalg.inv(H_true) @ np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])
    X, Y = Q[0] / Q[2], Q[1] / Q[2]
    inside = (X >= 0) & (X < 240) & (Y >= 0) & (Y < 180)
    check = ((np.floor(X / TILE) + np.floor(Y / TILE)) % 2 == 0)
    joint = (np.abs(((X + TILE / 2) % TILE) - TILE / 2) < 0.8) | (np.abs(((Y + TILE / 2) % TILE) - TILE / 2) < 0.8)
    val = np.where(check, 205, 120).astype(float)
    val[joint] = 40
    val[~inside] = 235
    img = np.stack([val, val * 0.95, val * 0.85], axis=-1).reshape(H_img, W_img, 3).astype(np.uint8)
else:
    im = Image.open(IMG_PATH).convert("RGB")
    if im.width > MAX_WIDTH:
        im = im.resize((MAX_WIDTH, int(im.height * MAX_WIDTH / im.width)), Image.LANCZOS)
    img = np.asarray(im)

print("Imagen:", img.shape, img.dtype)
plt.figure(figsize=(9, 6)); plt.imshow(img); plt.title("Imagen original (con rejilla para leer coordenadas)")
plt.grid(True, alpha=.4); plt.show()

## 2. Puntos origen: las 4 esquinas de un rectángulo real

Marca en la foto las **4 esquinas de un rectángulo que en el mundo real sea un rectángulo** (por ejemplo, las esquinas de varias baldosas formando un rectángulo) y anota su ancho y alto reales en cm en la configuración.

Orden obligatorio: **TL → TR → BR → BL** (en el sentido de las agujas del reloj, empezando por arriba a la izquierda).

> **Trasfondo (Clase 1, "Homografías" y "Creando homografías").** Una homografía tiene 9 parámetros pero, al ser homogénea (solo importa hasta un factor de escala), **8 grados de libertad**. Cada correspondencia punto↔punto aporta 2 ecuaciones (una por coordenada). Por tanto **4 puntos (8 ecuaciones) determinan la $H$**. Por eso el enunciado pide exactamente 4 puntos y por eso conviene que **no haya 3 colineales** (si lo hubiera, las ecuaciones dejarían de ser independientes).
>
> **Consejo práctico:** elige un rectángulo *grande* (que ocupe buena parte de la imagen) y marca con zoom. Un error de ±1 px en una esquina cercana se amplifica en la zona lejana de la imagen rectificada.

In [ ]:
# ---------- PUNTOS ORIGEN (píxeles de la imagen original, orden TL, TR, BR, BL) ----------
if DEMO:
    rng = np.random.default_rng(0)
    PTS_IMG = cam + rng.normal(0, 1.0, cam.shape)           # esquinas "marcadas a mano" con ±1 px de error
else:
    PTS_IMG = np.array([
        [0.0, 0.0],   # TL  <- sustituir por tus coordenadas
        [0.0, 0.0],   # TR
        [0.0, 0.0],   # BR
        [0.0, 0.0],   # BL
    ])
    assert PTS_IMG.any(), "Rellena PTS_IMG con las coordenadas (x, y) en píxeles de tus 4 esquinas"

print("Puntos origen (px):\n", PTS_IMG)
plt.figure(figsize=(9, 6)); plt.imshow(img)
closed = np.vstack([PTS_IMG, PTS_IMG[:1]])
plt.plot(closed[:, 0], closed[:, 1], "r-o")
for i, (x, y) in enumerate(PTS_IMG):
    plt.text(x + 8, y - 8, ["TL", "TR", "BR", "BL"][i], color="yellow", fontsize=12, weight="bold")
plt.title("Rectángulo de referencia marcado en la imagen"); plt.show()

## 3. Puntos destino: escala proporcional a las dimensiones físicas

En la vista cenital, el rectángulo debe tener la **misma proporción** que en el mundo real. Si el rectángulo mide $W \times H$ cm y elegimos una escala de `PX_PER_CM` píxeles por cm, las esquinas destino son:

$$
(x',y') \in \{(m,m),\ (m+W,m),\ (m+W,m+H),\ (m,m+H)\}\cdot s
$$

con $s$ = `PX_PER_CM` y $m$ = `MARGIN_CM` (un margen para ver también el suelo que rodea al rectángulo).

> **Trasfondo (Clase 2, "Metrología plana").** Al elegir la escala en px/cm, la imagen rectificada deja de ser solo "bonita": pasa a ser **métrica**. Una distancia de $d$ píxeles en la salida equivale a $d/s$ cm reales. Esa es la idea de la metrología plana: con un plano y una referencia de medidas conocidas, una sola foto permite medir.

In [ ]:
corners_cm = np.array([[0, 0], [REAL_W_CM, 0], [REAL_W_CM, REAL_H_CM], [0, REAL_H_CM]], dtype=float)
PTS_DST = (corners_cm + MARGIN_CM) * PX_PER_CM

OUT_W = int(round((REAL_W_CM + 2 * MARGIN_CM) * PX_PER_CM))
OUT_H = int(round((REAL_H_CM + 2 * MARGIN_CM) * PX_PER_CM))

print("Puntos destino (px):\n", PTS_DST)
print(f"Imagen de salida: {OUT_W} x {OUT_H} px  ({1/PX_PER_CM:.3f} cm/px)")

## 4. Cálculo de la homografía $H$: DLT + SVD + normalización de Hartley

Buscamos $H$ tal que $\mathbf{x}'_i \sim H\,\mathbf{x}_i$ (imagen → vista cenital).

> **Trasfondo (Clase 2, "Algoritmo DLT").**
>
> 1. **Formulación lineal.** Que $\mathbf{x}'$ y $H\mathbf{x}$ sean proporcionales equivale a que su producto vectorial sea nulo: $\mathbf{x}'_i \times (H\mathbf{x}_i) = \mathbf{0}$. Esto da, por cada punto, **2 ecuaciones linealmente independientes** en las 9 incógnitas $\mathbf{h}=(h_{11},\dots,h_{33})$. Para un punto $(x,y)\to(x',y')$:
>
> $$
\begin{bmatrix} -x & -y & -1 & 0 & 0 & 0 & x'x & x'y & x' \\ 0 & 0 & 0 & -x & -y & -1 & y'x & y'y & y' \end{bmatrix}\mathbf{h}=\mathbf{0}
$$
>
> 2. **Sistema homogéneo $A\mathbf{h}=\mathbf{0}$ (8×9 con 4 puntos).** Con ruido no hay solución exacta no nula. La **SVD** $A=U\Sigma V^T$ da la mejor: el vector que minimiza $\|A\mathbf{h}\|$ con $\|\mathbf{h}\|=1$ es la **última columna de $V$** (la asociada al menor valor singular). Se reordena en $3\times3$.
>
> 3. **Por qué hay que normalizar (Hartley).** Con píxeles del orden de $10^3$, los productos $x\,x'$ son del orden de $10^6$ mientras otros términos valen 1. El número de condición $\sigma_1/\sigma_n$ se dispara ($\gg10^8$), la SVD pierde precisión y el ruido de ±0.5 px puede producir errores de medida del 30–50 %. La normalización de Hartley lo evita en 3 pasos: **escalar → resolver → desescalar**.
>
> $$
T=\begin{bmatrix}s&0&-s\bar x\\0&s&-s\bar y\\0&0&1\end{bmatrix},\quad s=\frac{\sqrt2}{d},\quad d=\text{distancia media al centroide}
$$
>
> Se normalizan ambos conjuntos ($\hat{\mathbf{x}}=T\mathbf{x}$, $\hat{\mathbf{x}}'=T'\mathbf{x}'$), se resuelve $\hat H$ por SVD y se desnormaliza: $H=(T')^{-1}\,\hat H\,T$, y finalmente $H\leftarrow H/h_{33}$.
>
> Tras Hartley, el centroide queda en el origen y la distancia media es $\sqrt2$, así que todos los elementos de $A$ rondan el valor 1 y la SVD trabaja en su zona de máxima precisión.

In [ ]:
def hartley_normalization(pts):
    # Matriz T (3x3) que lleva el centroide al origen y deja la distancia media al origen en sqrt(2).
    pts = np.asarray(pts, dtype=np.float64)
    centroide = pts.mean(axis=0)
    d_media = np.linalg.norm(pts - centroide, axis=1).mean()
    s = np.sqrt(2) / d_media
    T = np.array([[s, 0, -s * centroide[0]],
                  [0, s, -s * centroide[1]],
                  [0, 0, 1.0]])
    pts_n = inhomog((T @ homog(pts).T).T)
    return pts_n, T

def build_A(src, dst):
    # Construye la matriz A (2N x 9) del sistema DLT  A h = 0
    A = []
    for (x, y), (xp, yp) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, xp * x, xp * y, xp])
        A.append([0, 0, 0, -x, -y, -1, yp * x, yp * y, yp])
    return np.array(A, dtype=np.float64)

def homography_dlt(src, dst, hartley=True, verbose=False):
    # Homografía src->dst con DLT + SVD. Con hartley=True aplica la normalización isótropa.
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    if hartley:
        src_n, T  = hartley_normalization(src)
        dst_n, Tp = hartley_normalization(dst)
    else:
        src_n, dst_n = src, dst
        T = Tp = np.eye(3)

    A = build_A(src_n, dst_n)
    U, S, Vt = np.linalg.svd(A)
    h_norm = Vt[-1]                                  # última fila de V^T = última columna de V
    H_norm = h_norm.reshape(3, 3)
    H = np.linalg.inv(Tp) @ H_norm @ T               # desnormalización
    H = H / H[2, 2]                                  # h33 = 1

    if verbose:
        print(f"valores singulares de A: {S}")
        print(f"número de condición (considerando los 8 valores no nulos): {S[0] / S[min(7, len(S)-1)]:.3e}")
    return H, S

H, S = homography_dlt(PTS_IMG, PTS_DST, hartley=True, verbose=True)
H_sin, S_sin = homography_dlt(PTS_IMG, PTS_DST, hartley=False, verbose=True)

print("\nH (con Hartley), imagen -> vista cenital:\n", H)

# Comprobación: H debe llevar las 4 esquinas a sus destinos
err_reproj = np.linalg.norm(apply_H(H, PTS_IMG) - PTS_DST, axis=1)
print("\nError de reproyección de las 4 esquinas (px):", err_reproj)

**Lectura de los resultados (para la entrevista).**

* El **error de reproyección** de las 4 esquinas debe ser ≈ 0 (≲ $10^{-9}$): con exactamente 4 puntos el sistema está *exactamente determinado* y la $H$ los lleva a su sitio.
* Compara los **números de condición** con y sin Hartley: la diferencia de órdenes de magnitud es la justificación numérica de la normalización (Clase 2, "La catástrofe numérica sin normalización").
* Estructura de $H$ por bloques (Clase 2, "Anatomía de una homografía"): el bloque $A_{2\times2}$ (rotación/escala/cizalla), la traslación $\mathbf t$ y, sobre todo, el vector proyectivo $\mathbf v^T=(h_{31},h_{32})$, que es el que **rompe el paralelismo**: si $\mathbf v=\mathbf 0$ la transformación sería afín y no corregiría la perspectiva.

## 5. Warp propio (mapeo inverso + bilineal) y verificación del paralelismo

> **Trasfondo (Sesión 1, notebook de apoyo: "Deformación por mapeo inverso").** Si recorres la imagen *origen* y empujas cada píxel con $H$, la imagen de salida queda con **huecos** (varios píxeles de origen caen en el mismo destino, y otros destinos no reciben ninguno). La solución es el **mapeo inverso (backward mapping)**: para cada píxel de la imagen de *salida* se calcula de dónde viene con $\mathbf p_{src}\sim H^{-1}\mathbf p_{dst}$, se divide entre la coordenada homogénea $w$ y se **interpola** el valor. Aquí se usa interpolación **bilineal** (media ponderada de los 4 vecinos) y está **vectorizado** con NumPy.

Como $H$ lleva *imagen → cenital*, el warp necesita $H^{-1}$.

In [ ]:
t0 = time.perf_counter()
rect_np = warp_bilinear(img, np.linalg.inv(H), OUT_W, OUT_H)
t_np = time.perf_counter() - t0
print(f"Warp NumPy (bilineal, vectorizado): {t_np:.3f} s -> salida {rect_np.shape}")

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img);     ax[0].set_title("Original (perspectiva)")
ax[1].imshow(rect_np); ax[1].set_title("Vista cenital (NumPy)")
plt.tight_layout(); plt.show()

### 5.1 Verificación empírica del paralelismo

Para demostrar que el paralelismo se ha recuperado, trabajamos con **rectas en coordenadas homogéneas** (Clase 1, "Geometría del plano usando homogéneas"):

* La recta que pasa por dos puntos $P,Q$ es $\mathbf l=P\times Q$.
* La intersección de dos rectas es $\mathbf P=\mathbf l_1\times\mathbf l_2$.
* Si dos rectas son **paralelas**, su intersección es un **punto del infinito** ($w=0$), que en coordenadas cartesianas queda "infinitamente lejos".

Procedimiento:

1. Marca varios puntos (≥ 2, mejor 4-6) sobre **cada junta** de dos familias de juntas: familia A (las que fugan en una dirección) y familia B (las que cortan a las anteriores).
2. En la imagen original, ajusta una recta a cada junta por mínimos cuadrados totales (SVD) y calcula los **ángulos entre juntas de la misma familia** y su **punto de fuga** (intersección).
3. Transforma esos mismos puntos con $H$ (una homografía lleva rectas en rectas) y repite las medidas sobre la vista cenital: las juntas de una misma familia deben tener ángulo ≈ 0° y su intersección debe irse al infinito; además, las dos familias deben formar ≈ 90° (las baldosas son cuadradas/rectangulares).

> En una homografía las rectas se transforman con $\mathbf l' = H^{-T}\mathbf l$ (Clase 1). Aquí transformamos los puntos y reajustamos, que es equivalente y permite incluir el ruido de marcado.

In [ ]:
# ---------- PUNTOS SOBRE LAS JUNTAS (píxeles de la imagen original) ----------
# Cada junta = lista de puntos (x, y) que están sobre ella. Mínimo 2 por junta, mejor 4-6.
if DEMO:
    def junta_vertical(xc):                                  # junta x = xc cm (familia A)
        return apply_H(H_true, np.array([[xc, y] for y in (5, 50, 95, 140, 175)], float)) + rng.normal(0, .7, (5, 2))
    def junta_horizontal(yc):                                # junta y = yc cm (familia B)
        return apply_H(H_true, np.array([[x, yc] for x in (5, 60, 120, 180, 235)], float)) + rng.normal(0, .7, (5, 2))
    FAMILIA_A = [junta_vertical(60), junta_vertical(120), junta_vertical(180)]
    FAMILIA_B = [junta_horizontal(60), junta_horizontal(120)]
else:
    FAMILIA_A = [   # juntas que fugan en una dirección (p. ej. las "verticales" de la foto)
        # np.array([[x1, y1], [x2, y2], [x3, y3]]),
        # np.array([[...]]),
    ]
    FAMILIA_B = [   # juntas de la otra dirección
        # np.array([[...]]),
    ]
    assert len(FAMILIA_A) >= 2 and len(FAMILIA_B) >= 2, "Rellena al menos 2 juntas por familia"

def fit_line(pts):
    # Recta (a,b,c) con a x + b y + c = 0 ajustada por mínimos cuadrados totales (SVD).
    pts = np.asarray(pts, float)
    c = pts.mean(axis=0)
    _, _, Vt = np.linalg.svd(pts - c)
    d = Vt[0]                                  # dirección de la recta
    n = np.array([-d[1], d[0]])                # normal
    return np.array([n[0], n[1], -n @ c]), d

def angle_between(d1, d2):
    # Ángulo agudo (grados) entre dos direcciones
    cosang = abs(d1 @ d2) / (np.linalg.norm(d1) * np.linalg.norm(d2))
    return np.degrees(np.arccos(np.clip(cosang, -1, 1)))

def analyze(familia, nombre):
    # Ángulos entre pares de juntas de la misma familia y distancia de su intersección
    lines = [fit_line(p) for p in familia]
    angs, dist = [], []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            angs.append(angle_between(lines[i][1], lines[j][1]))
            P = np.cross(lines[i][0], lines[j][0])             # intersección = l1 x l2 (Clase 1)
            dist.append(np.inf if abs(P[2]) < 1e-12 else np.hypot(P[0] / P[2], P[1] / P[2]))
    return lines, np.array(angs), np.array(dist)

def informe(FA, FB, etiqueta):
    LA, angA, dA = analyze(FA, "A")
    LB, angB, dB = analyze(FB, "B")
    cross = [angle_between(la[1], lb[1]) for la in LA for lb in LB]
    print(f"--- {etiqueta} ---")
    print(f"Familia A: ángulo máx. entre juntas = {angA.max():8.4f}°   | intersección más cercana al origen: {dA.min():12.1f} px")
    print(f"Familia B: ángulo máx. entre juntas = {angB.max():8.4f}°   | intersección más cercana al origen: {dB.min():12.1f} px")
    print(f"Ángulo entre familias A y B: media = {np.mean(cross):.3f}° (rango {np.min(cross):.2f}°-{np.max(cross):.2f}°)")
    return angA, angB, cross

FA_rect = [apply_H(H, p) for p in FAMILIA_A]
FB_rect = [apply_H(H, p) for p in FAMILIA_B]

res_orig = informe(FAMILIA_A, FAMILIA_B, "IMAGEN ORIGINAL (perspectiva)")
res_rect = informe(FA_rect, FB_rect, "VISTA CENITAL (rectificada)")

In [ ]:
# ---------- Visualización: juntas ajustadas sobre las dos imágenes ----------
def dibujar(ax, fondo, FA, FB, titulo):
    ax.imshow(fondo)
    for fam, col in ((FA, "red"), (FB, "cyan")):
        for p in fam:
            line, d = fit_line(p)
            c = np.asarray(p).mean(axis=0)
            t = np.array([-3000, 3000])[:, None]
            seg = c + t * d
            ax.plot(seg[:, 0], seg[:, 1], color=col, lw=1.2)
            ax.plot(np.asarray(p)[:, 0], np.asarray(p)[:, 1], ".", color="yellow", ms=7)
    ax.set_xlim(0, fondo.shape[1]); ax.set_ylim(fondo.shape[0], 0)
    ax.set_title(titulo)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
dibujar(ax[0], img,     FAMILIA_A, FAMILIA_B, "Original: las juntas CONVERGEN")
dibujar(ax[1], rect_np, FA_rect,   FB_rect,   "Rectificada: las juntas son PARALELAS")
plt.tight_layout(); plt.show()

**Lectura de los resultados.**

* En la imagen original, el ángulo entre juntas de una misma familia es de varios grados y sus rectas se cortan en un **punto de fuga finito** (a cientos/miles de píxeles).
* En la vista cenital, ese ángulo cae a ≈ 0° y la intersección se aleja a distancias enormes (≳ $10^4$–$10^6$ px), es decir, **el punto de fuga se ha enviado al infinito**: justo lo que debe hacer una rectificación (Clase 1, "Puntos y rectas en el infinito").
* Las dos familias pasan a formar ≈ 90°. Eso comprueba, además del paralelismo, que la rectificación es **métrica** y no solo afín.
* Qué lo explica: la $H$ calculada con las 4 esquinas deshace la perspectiva de *todo* el plano del suelo (no solo del rectángulo), porque la relación plano↔imagen es una homografía única.

## 6. Comparación con OpenCV

A partir de aquí (y solo aquí) se usa OpenCV como **referencia** para contrastar el resultado propio.

* `cv2.findHomography(src, dst, 0)` (mínimos cuadrados, sin RANSAC) y `cv2.getPerspectiveTransform` (exacta para 4 puntos).
* `cv2.warpPerspective` con interpolación bilineal.

> **Trasfondo (Notebook "RepasoHomografías").** Con exactamente **4 puntos** el sistema es determinado y NumPy-DLT y OpenCV deben coincidir hasta precisión de máquina. Con más de 4 puntos el sistema es **sobredeterminado** y pueden aparecer diferencias pequeñas porque cada método minimiza un error distinto (el DLT minimiza un *error algebraico*; OpenCV, tras la estimación inicial, puede refinar con otro criterio).

In [ ]:
import cv2
print("OpenCV", cv2.__version__)

src32 = PTS_IMG.astype(np.float32)
dst32 = PTS_DST.astype(np.float32)

# --- Homografías de OpenCV ---
H_cv_fh, _ = cv2.findHomography(src32, dst32, 0)
H_cv_fh = H_cv_fh / H_cv_fh[2, 2]
H_cv_gp = cv2.getPerspectiveTransform(src32, dst32)
H_cv_gp = H_cv_gp / H_cv_gp[2, 2]

print("H propia (NumPy):\n", H)
print("\nH cv2.findHomography:\n", H_cv_fh)
print("\nDiferencia máx. |H_np - H_findHomography|     =", np.abs(H - H_cv_fh).max())
print("Diferencia máx. |H_np - H_getPerspectiveTransform| =", np.abs(H - H_cv_gp).max())

# --- Warp de OpenCV ---
t0 = time.perf_counter()
rect_cv = cv2.warpPerspective(img, H_cv_fh, (OUT_W, OUT_H), flags=cv2.INTER_LINEAR)
t_cv = time.perf_counter() - t0

# --- Comparación de imágenes ---
diff = np.abs(rect_np.astype(np.float64) - rect_cv.astype(np.float64))
mse = (diff ** 2).mean()
psnr = np.inf if mse == 0 else 10 * np.log10(255 ** 2 / mse)
pix_dist = (diff.max(axis=-1) > 2).mean() * 100          # % de píxeles que difieren en más de 2 niveles

print(f"\nTiempo warp: NumPy {t_np:.3f} s  vs  OpenCV {t_cv:.4f} s")
print(f"Diferencia media absoluta: {diff.mean():.4f} niveles  |  máxima: {diff.max():.0f}  |  PSNR: {psnr:.2f} dB")
print(f"Píxeles con diferencia > 2 niveles: {pix_dist:.3f} %")

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].imshow(rect_np);                       ax[0].set_title("NumPy (propio)")
ax[1].imshow(rect_cv);                       ax[1].set_title("OpenCV warpPerspective")
im = ax[2].imshow(diff.max(axis=-1), cmap="magma"); ax[2].set_title("|NumPy - OpenCV| (máx. canal)")
plt.colorbar(im, ax=ax[2], fraction=0.046); plt.tight_layout(); plt.show()

In [ ]:
# Medición del paralelismo también sobre la imagen rectificada por OpenCV (mismos puntos, H de OpenCV)
FA_cv = [apply_H(H_cv_fh, p) for p in FAMILIA_A]
FB_cv = [apply_H(H_cv_fh, p) for p in FAMILIA_B]
res_cv = informe(FA_cv, FB_cv, "VISTA CENITAL con H de OpenCV")

print("\n=== RESUMEN ===")
print(f"Ángulo máx. entre juntas de la familia A:  original {res_orig[0].max():.3f}°  ->  NumPy {res_rect[0].max():.4f}°  |  OpenCV {res_cv[0].max():.4f}°")
print(f"Ángulo máx. entre juntas de la familia B:  original {res_orig[1].max():.3f}°  ->  NumPy {res_rect[1].max():.4f}°  |  OpenCV {res_cv[1].max():.4f}°")
print(f"Ángulo medio entre familias A y B:         original {np.mean(res_orig[2]):.2f}°  ->  NumPy {np.mean(res_rect[2]):.3f}°  |  OpenCV {np.mean(res_cv[2]):.3f}°")

> **Fin de la plantilla (apartados 0-6).**
> Pendiente para completar: apartado 7 (variación opcional: p. ej. más de 4 puntos y mínimos cuadrados, o medir una distancia real sobre la vista cenital) y el informe en PDF.